# Практична робота 04: Проєктування рішення навколо фіксованого класифікатора

## Набір даних Hill-Valley (UCI)

Мета: зафіксувати модель, отримати OOF-бали, підготувати основу для аналізу порогів.

**Увага:** на цьому етапі ми не виконуємо повний цикл (сітка порогів, калібрування, політика, test).
Лише фіксація протоколу та отримання OOF-балів.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

### 1. Завантаження даних

Використовуємо `ucimlrepo` для отримання набору Hill-Valley (id=166).
Обираємо версію **без шуму** (Hill_Valley_without_noise) для чистоти експерименту.

In [3]:
import pandas as pd

# 1. Downloading file. adding header=None, 
hill_valley = pd.read_csv('Hill_Valley_without_noise_Training.data', header=None)

# 2. dividing data:
# iloc[:, :-1] takes all except last (cause it X)
X = hill_valley.iloc[:, :-1]

# iloc[:, -1] takes all except last (cause it y)
y = hill_valley.iloc[:, -1]

# 3. Outputing data
print('Форма X:', X.shape)
print('Форма y:', y.shape)
print('Розподіл класів:')
print(y.value_counts())
print('\nПерші 5 ознак:')
print(X.head())


Форма X: (607, 100)
Форма y: (607,)
Розподіл класів:
100
0        305
1        301
class      1
Name: count, dtype: int64

Перші 5 ознак:
            0            1            2            3            4   \
0           X1           X2           X3           X4           X5   
1  1317.265789  1315.220951  1312.770581  1309.834252  1306.315588   
2  7329.967624  7379.907443  7441.799231  7518.503422  7613.565031   
3  809.4214096  809.7801194  810.2071911  810.7156529   811.321016   
4  45334.20888  45334.21356  45334.21906   45334.2255  45334.23305   

            5            6            7            8            9   ...  \
0           X6           X7           X8           X9          X10  ...   
1  1302.099102  1297.046401  1290.991646  1283.736109  1275.041652  ...   
2  7731.377492  7877.385707  8058.337694  8282.596458  8560.526497  ...   
3  812.0417476  812.8998341  813.9214524   815.137768  816.5858856  ...   
4  45334.24191   45334.2523  45334.26448  45334.27876  45334.29552

### 2. Фіксація train/test split та random_state

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split


hill_valley = pd.read_csv('Hill_Valley_without_noise_Training.data') 


X = hill_valley.iloc[:, :-1]
y = hill_valley.iloc[:, -1]


RANDOM_STATE = 42
TEST_SIZE = 0.2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Розподіл класів у train:')
print(y_train.value_counts(normalize=True))
print('Розподіл класів у test:')
print(y_test.value_counts(normalize=True))


Train: (484, 100) Test: (122, 100)
Розподіл класів у train:
class
0    0.504132
1    0.495868
Name: proportion, dtype: float64
Розподіл класів у test:
class
1    0.5
0    0.5
Name: proportion, dtype: float64


### 3. Фіксована модель та Pipeline

Обираємо **LogisticRegression** як базовий класифікатор.
Preprocessing: StandardScaler у Pipeline.
Гіперпараметри: `C=1.0`, `solver='lbfgs'`, `max_iter=1000`.

Якщо у вашій лабораторній роботі №2 використовувалась інша модель (наприклад, RandomForest),
замініть її тут, але зафіксуйте всі параметри.

In [5]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=RANDOM_STATE))
])

print('Pipeline зафіксовано:')
print(pipeline)

Pipeline зафіксовано:
Pipeline(steps=[('scaler', StandardScaler()),
                ('clf', LogisticRegression(max_iter=1000, random_state=42))])


### 4. Стратифікована крос-валідація та OOF-бали

Використовуємо `StratifiedKFold` з 5 фолдами.
Спосіб отримання балу: `predict_proba` (ймовірність класу 1).

Для кожного об'єкта тренувального набору отримуємо прогноз від моделі,
яка не навчалася на цьому об'єкті.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Отримуємо OOF-ймовірності для всього train набору
oof_scores = cross_val_predict(
    pipeline, X_train, y_train,
    cv=cv,
    method='predict_proba',
    n_jobs=-1
)[:, 1]  # ймовірність класу 1 (hill)

# Перевіряємо, що кожен об'єкт має рівно один прогноз
assert len(oof_scores) == len(X_train), 'Кількість OOF-прогнозів не збігається з кількістю об\'єктів'

print('OOF-бали отримано. Приклад перших 10:')
print(oof_scores[:10])

OOF-бали отримано. Приклад перших 10:
[0.50952585 0.85534823 0.40540655 0.5083684  0.50832839 0.49829611
 0.52178756 0.50138154 0.49823641 0.50817557]


### 5. Формування таблиці OOF

Створюємо DataFrame з полями: `object_id`, `y_true`, `score`, `fold`.
`object_id` — індекс об'єкта у вихідному train наборі (0..N-1).
`fold` — номер фолду, у якому об'єкт був у валідаційній частині.

In [7]:
# Визначаємо fold для кожного об'єкта
fold_ids = np.zeros(len(X_train), dtype=int)
for fold_idx, (_, val_idx) in enumerate(cv.split(X_train, y_train)):
    fold_ids[val_idx] = fold_idx

oof_df = pd.DataFrame({
    'object_id': np.arange(len(X_train)),
    'y_true': y_train.values.ravel(),
    'score': oof_scores,
    'fold': fold_ids
})

print('Форма OOF-таблиці:', oof_df.shape)
print(oof_df.head(10))
print('\nРозподіл по фолдах:')
print(oof_df['fold'].value_counts().sort_index())

Форма OOF-таблиці: (484, 4)
   object_id  y_true     score  fold
0          0       1  0.509526     0
1          1       1  0.855348     0
2          2       0  0.405407     1
3          3       1  0.508368     1
4          4       0  0.508328     1
5          5       0  0.498296     2
6          6       0  0.521788     3
7          7       0  0.501382     4
8          8       0  0.498236     2
9          9       1  0.508176     2

Розподіл по фолдах:
fold
0    97
1    97
2    97
3    97
4    96
Name: count, dtype: int64


### 6. Базова оцінка OOF-балів

Обчислюємо ROC-AUC та accuracy за стандартним порогом 0.5.
Це проміжна оцінка, яка не є фінальною метрикою рішення.

In [8]:
roc_auc = roc_auc_score(oof_df['y_true'], oof_df['score'])
y_pred_default = (oof_df['score'] >= 0.5).astype(int)
acc = accuracy_score(oof_df['y_true'], y_pred_default)

print(f'OOF ROC-AUC: {roc_auc:.4f}')
print(f'OOF Accuracy (порог 0.5): {acc:.4f}')

OOF ROC-AUC: 0.7814
OOF Accuracy (порог 0.5): 0.6694


### 7. Збереження OOF-таблиці

Зберігаємо `oof_results.csv` у директорію `../data/` для подальшого аналізу.

In [9]:
import os
os.makedirs('data', exist_ok=True)
oof_df.to_csv('data/oof_results.csv', index=False)
print('OOF-таблицю збережено у ../data/oof_results.csv')

OOF-таблицю збережено у ../data/oof_results.csv


---

## Примітки

- `score` у таблиці — це ймовірність класу 1 (hill), отримана через `predict_proba`.
- `fold` вказує, у якому фолді об'єкт був у валідаційній вибірці.
- Наступні кроки (сітка порогів, аналіз FP/FN, калібрування) будуть виконані в окремих комірках або наступних ноутбуках.

---

# 3. Контрольна політика (baseline)

## 3.1. Вибір початкового порогу

Оскільки для отримання балів використано `predict_proba` моделі `LogisticRegression`,
значення `score` можна інтерпретувати як **калібровану ймовірність** (наближено).

Тому для **baseline-політики** обираємо стандартний поріг:

$$t_0 = 0.5$$

**Правило прийняття рішення:**
- якщо `score >= 0.5` → `ŷ = 1` (пагорб) → запускається дія
- якщо `score < 0.5` → `ŷ = 0` (долина) → дія не запускається

**Важливо:** test set на цьому етапі **не використовується**.
Усі обчислення — лише на OOF-балах тренувального набору.

In [10]:
# Фіксуємо baseline-поріг
t0 = 0.5

# Бінарні рішення за baseline-політикою
y_pred_baseline = (oof_df['score'] >= t0).astype(int)

print(f'Baseline поріг t0 = {t0}')
print(f'Кількість позитивних рішень (ŷ=1): {y_pred_baseline.sum()}')
print(f'Кількість негативних рішень (ŷ=0): {(1 - y_pred_baseline).sum()}')

Baseline поріг t0 = 0.5
Кількість позитивних рішень (ŷ=1): 340
Кількість негативних рішень (ŷ=0): 144


## 3.2. Матриця помилок (confusion matrix)

Обчислюємо матрицю помилок на OOF-прогнозах.
Порядок класів: `[0, 1]`, тобто:
- TN — правильно передбачена долина
- FP — хибно передбачений пагорб
- FN — пропущений пагорб
- TP — правильно передбачений пагорб

In [11]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

cm = confusion_matrix(oof_df['y_true'], y_pred_baseline, labels=[0, 1])

tn, fp, fn, tp = cm.ravel()

print('Confusion matrix (рядки = true, стовпці = pred):')
print(pd.DataFrame(cm,
                   index=['True 0 (valley)', 'True 1 (hill)'],
                   columns=['Pred 0 (valley)', 'Pred 1 (hill)']))
print()
print(f'TN = {tn}')
print(f'FP = {fp}')
print(f'FN = {fn}')
print(f'TP = {tp}')

Confusion matrix (рядки = true, стовпці = pred):
                 Pred 0 (valley)  Pred 1 (hill)
True 0 (valley)              114            130
True 1 (hill)                 30            210

TN = 114
FP = 130
FN = 30
TP = 210


## 3.3. Метрики baseline-політики

Обчислюємо:
- **Precision** — частка правильних серед передбачених `ŷ=1`
- **Recall** — частка виявлених пагорбів серед усіх справжніх пагорбів
- **F1** — гармонійне середнє Precision і Recall
- **q(t)** — частка позитивних рішень (доля об'єктів, для яких запускається дія)
- **Дії на 1000 об'єктів** — скільки разів система запустить дію на кожні 1000 об'єктів

In [12]:
precision = precision_score(oof_df['y_true'], y_pred_baseline, pos_label=1)
recall = recall_score(oof_df['y_true'], y_pred_baseline, pos_label=1)
f1 = f1_score(oof_df['y_true'], y_pred_baseline, pos_label=1)

q_t = y_pred_baseline.mean()
actions_per_1000 = q_t * 1000

print('=== Метрики baseline-політики (t0 = 0.5) ===')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1        : {f1:.4f}')
print()
print(f'q(t0)     : {q_t:.4f}  ({q_t*100:.2f}%)')
print(f'Дій на 1000 об\'єктів: {actions_per_1000:.1f}')

=== Метрики baseline-політики (t0 = 0.5) ===
Precision : 0.6176
Recall    : 0.8750
F1        : 0.7241

q(t0)     : 0.7025  (70.25%)
Дій на 1000 об'єктів: 702.5


## 3.4. Прикладна інтерпретація baseline-політики

**Що означають отримані числа в контексті задачі:**

- **Precision** показує, наскільки можна довіряти позитивному рішенню `ŷ=1`.
  Якщо Precision = 0.95, то з 100 позначених «пагорбів» лише ~5 виявляться долинами.

- **Recall** показує, яку частку реальних пагорбів система взагалі виявляє.
  Якщо Recall = 0.85, то 15% пагорбів залишаються непоміченими (FN).

- **q(t0)** — це частка об'єктів, для яких запускається дія. Якщо q = 0.5,
  система позначає кожен другий об'єкт як «пагорб».

- **Дії на 1000 об'єктів** — практичний показник навантаження на систему
  (скільки разів буде активовано модуль подальшої обробки).

**Ключове питання для baseline:** чи влаштовує нас таке співвідношення FP і FN,
чи потрібно зміщувати поріг у бік зменшення одного з типів помилок?

Це питання буде вирішено в наступних розділах після побудови матриці вартостей.

---

# 4. Матриця вартостей

## 4.1. Таблиця наслідків для TN, FP, FN, TP

Для кожної комірки матриці помилок визначаємо:
1. **фактичний стан** об'єкта;
2. **рішення системи** (прогноз);
3. **операційну дію**, яку виконує система;
4. **можливий негативний або позитивний наслідок**;
5. **одиницю виміру** цього наслідку.

| Комірка | Факт | Рішення | Операційна дія | Наслідок | Одиниця виміру |
|---------|------|---------|----------------|----------|----------------|
| **TN** | Valley (0) | ŷ = 0 (valley) | Ділянка позначається як низовина; у маршруті враховується як безпечна | **Нейтральний / позитивний**: маршрут прокладено правильно, ресурси не витрачено даремно | Умовні одиниці витрат (0) |
| **FP** | Valley (0) | ŷ = 1 (hill) | Ділянка помилково позначається як пагорб; запускається додаткова обробка | **Негативний**: зайве ускладнення маршруту, витрати часу/ресурсів на обхід «пагорба», якого немає | Людино-години / хвилини обробки / умовні витрати |
| **FN** | Hill (1) | ŷ = 0 (valley) | Ділянка помилково позначається як низовина; маршрут прокладається напряму через неї | **Критично негативний**: потенційне зіткнення з рельєфом, пошкодження обладнання, загроза безпеці | Ймовірність аварії / вартість ремонту / ризик для життя |
| **TP** | Hill (1) | ŷ = 1 (hill) | Ділянка позначається як пагорб; маршрут коригується | **Позитивний**: безпечне проходження, коректна ідентифікація рельєфу | Умовні одиниці виграшу / збережені ресурси |

## 4.2. Визначення cFP та cFN

**Позначення:**
- $c_{FP}$ — вартість одного хибнопозитивного рішення (FP)
- $c_{FN}$ — вартість одного хибнонегативного рішення (FN)

**Ключове спостереження:**
У нашому прикладному контексті (аналіз рельєфу для автономного дрона / ГІС):

- **FP** → зайві витрати на обхід неіснуючого пагорба.
  Оцінка: **низька** (наприклад, 1 умовна одиниця).

- **FN** → пропущений пагорб → потенційне зіткнення.
  Оцінка: **висока** (наприклад, 10 умовних одиниць через ризик аварії).

**Відношення:**
$$\frac{c_{FN}}{c_{FP}} \approx 10$$

**Тип оцінки:**
- cFP — **суто сценарна** (експертна оцінка витрат на додаткову обробку).
- cFN — **експертна / нормативна** (базується на оцінці ризику для безпеки).

**Джерело / аргумент:**
- Для cFP: середній час оператора на перевірку одного об'єкта × вартість години.
- Для cFN: оцінка збитків від аварії (ремонт + простій + можливі штрафи).
- **ШІ не є джерелом реальної вартості** — це лише сценарне припущення для демонстрації методу.

**Важливо:** точні значення cFP і cFN мають бути узгоджені з замовником
або експертом предметної області. Тут вони наведені як **робоча гіпотеза**.

In [13]:
# Сценарні вартості (умовні одиниці)
c_FP = 1.0    # вартість одного FP
c_FN = 10.0   # вартість одного FN

# Очікувана вартість baseline-політики на OOF
expected_cost_baseline = fp * c_FP + fn * c_FN

# Нормалізована вартість на один об'єкт
cost_per_object = expected_cost_baseline / len(oof_df)

print('=== Оцінка вартості baseline-політики (t0 = 0.5) ===')
print(f'c_FP = {c_FP}, c_FN = {c_FN}')
print(f'FP = {fp}, FN = {fn}')
print(f'Очікувана сумарна вартість: {expected_cost_baseline:.1f} умовних одиниць')
print(f'Вартість на один об\'єкт: {cost_per_object:.4f}')
print()
print(f'Відношення c_FN / c_FP = {c_FN / c_FP:.1f}')
print()
print('Примітка: ці значення будуть використані для побудови сітки порогів')
print('та вибору політики, що мінімізує очікувану вартість.')

=== Оцінка вартості baseline-політики (t0 = 0.5) ===
c_FP = 1.0, c_FN = 10.0
FP = 130, FN = 30
Очікувана сумарна вартість: 430.0 умовних одиниць
Вартість на один об'єкт: 0.8884

Відношення c_FN / c_FP = 10.0

Примітка: ці значення будуть використані для побудови сітки порогів
та вибору політики, що мінімізує очікувану вартість.


## 4.3. Проміжний висновок

На цьому етапі ми маємо:

1. **Зафіксовану baseline-політику** з порогом `t0 = 0.5`.
2. **Метрики baseline**: Precision, Recall, F1, q(t), дії на 1000 об'єктів.
3. **Матрицю вартостей** з описаними наслідками TN, FP, FN, TP.
4. **Сценарні оцінки** cFP та cFN (з відношенням ~1:10).

**Наступні кроки:**
- Побудова **сітки порогів** `t ∈ [0, 1]`.
- Для кожного порогу — обчислення FP, FN, q(t) та очікуваної вартості.
- Вибір порогу, що мінімізує очікувану вартість (або задовольняє обмеження на Recall / Precision).
- **Калібрування** (за потреби) для покращення інтерпретації ймовірностей.
- **Одноразовий test** — лише після фіксації фінальної політики.

**Test set на цьому етапі не використовувався.**

In [16]:
# Зберігаємо результати baseline та вартості для подальшого аналізу
baseline_results = {
    'threshold': t0,
    'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'q_t': q_t,
    'actions_per_1000': actions_per_1000,
    'c_FP': c_FP,
    'c_FN': c_FN,
    'expected_cost': expected_cost_baseline,
    'cost_per_object': cost_per_object
}

baseline_df = pd.DataFrame([baseline_results])
baseline_df.to_csv('data/baseline_policy.csv', index=False)
print('Результати baseline збережено у ../data/baseline_policy.csv')
print(baseline_df.T)

Результати baseline збережено у ../data/baseline_policy.csv
                           0
threshold           0.500000
TN                114.000000
FP                130.000000
FN                 30.000000
TP                210.000000
precision           0.617647
recall              0.875000
f1                  0.724138
q_t                 0.702479
actions_per_1000  702.479339
c_FP                1.000000
c_FN               10.000000
expected_cost     430.000000
cost_per_object     0.888430
